# 🛢️ Oil Spill Detection — Module 2 Training
## Look-alike Discriminator & Bilge-Dump Characterization

**Pipeline position:** Module 1 → **Module 2 (this notebook)** → Module 3 → Module 4

Trains the **Random Forest look-alike discriminator** on 12 tabular features extracted
from dark-patch connected components, then applies the bilge-dump morphology filter.

| Cell | Purpose |
|------|---------|
| Cell 0 | GPU verification |
| Cell 1 | Setup: repo clone + dependencies + Hugging Face connection |
| Cell 2 | Data symlinks (oil + lookalike) + dataset sanity check |
| Cell 3 | Download Module 1 checkpoint from HF Hub (for on-the-fly inference) |
| Cell 4 | Module 2 training — feature extraction + RF training + bilge filter |
| Cell 5 | Session summary + output collection |

> **Runtime estimate:** Feature extraction ~10–30 min · RF training ~2–5 min · Total < 1 hr on T4

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 0 — GPU VERIFICATION  (run first, always)
# ═══════════════════════════════════════════════════════════════════════════════
import subprocess, sys

result = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
     "--format=csv,noheader"],
    capture_output=True, text=True,
)
if result.returncode == 0:
    print(f"✅ GPU : {result.stdout.strip()}")
else:
    print("⚠️  No GPU detected — feature extraction will be slower but RF still trains.")

import torch
print(f"   CUDA: {torch.version.cuda}   PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"\n🟢 GPU ready — proceed to Cell 1.")
else:
    print(f"\n🟡 CPU only — proceed, but feature extraction will be slow.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1 — SETUP: Repo · Dependencies · Hugging Face Connection
#
# ─── CONFIGURE HERE ──────────────────────────────────────────────────────────
HF_REPO_ID = "RohithSheregar/oil-spill-models"   # same repo as Module 1
# ─────────────────────────────────────────────────────────────────────────────
import os, sys

# ── 1a. Clone / pull latest repo ─────────────────────────────────────────────
REPO_URL = "https://github.com/Rohith-Sheregar/Oil-Spill-Detection-New.git"
REPO_DIR = "/kaggle/working/repo"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print(f"✅ Repo: {os.getcwd()}")

# ── 1b. Install dependencies ──────────────────────────────────────────────────
print("\n📦 Installing deps...")
!pip install -q scikit-learn scikit-image scipy joblib shapely tifffile \
              pandas numpy matplotlib huggingface_hub \
              segmentation-models-pytorch albumentations imagecodecs
print("✅ Dependencies ready.")

# ── 1c. Load HF Token from Kaggle Secret ─────────────────────────────────────
HF_TOKEN = ""
print(f"\n🔑 Loading Hugging Face token from Kaggle Secret 'HF_TOKEN'...")
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("✅ Token loaded successfully.")
except Exception as _e:
    print(f"⚠️  Could not load secret 'HF_TOKEN': {_e}")
    print("   HF uploads will be DISABLED. Training will still run locally.")

# ── 1d. Hugging Face connection test ─────────────────────────────────────────
print("\n🧪 Testing Hugging Face connection...")
if not HF_REPO_ID:
    print("⚠️  HF_REPO_ID is empty — skipping HF test.")
elif not HF_TOKEN:
    print("⚠️  No HF_TOKEN — skipping HF test. Uploads disabled.")
else:
    try:
        import time
        from huggingface_hub import HfApi
        _api  = HfApi(token=HF_TOKEN)
        _user = _api.whoami()["name"]
        print(f"   ✅ Authenticated as: {_user}")
        _api.create_repo(repo_id=HF_REPO_ID, exist_ok=True, private=True)
        print(f"   ✅ Repo ready: https://huggingface.co/{HF_REPO_ID}")
        # Quick upload smoke test
        _test_name = f"_kaggle_hf_test_{int(time.time())}.txt"
        _test_path = f"/kaggle/working/{_test_name}"
        with open(_test_path, "w") as _tf:
            _tf.write("ok")
        _api.upload_file(path_or_fileobj=_test_path, path_in_repo=_test_name,
                         repo_id=HF_REPO_ID, commit_message="Kaggle Connection Test")
        print(f"   ✅ Upload OK  → {_test_name}")
        _api.delete_file(path_in_repo=_test_name, repo_id=HF_REPO_ID,
                         commit_message="Cleanup test file")
        print(f"   ✅ Cleanup OK → test file deleted")
        print(f"\n🟢 Hugging Face Hub connected! Auto-save is ACTIVE.")
    except Exception as _e:
        print(f"   ⚠️  HF connection failed: {_e}")
        HF_TOKEN = ""
        print("   HF uploads DISABLED for this session.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2 — DATA SYMLINKS + SANITY CHECK
# ─────────────────────────────────────────────────────────────────────────────
# Module 2 needs:
#   • train/oil       — oil-spill SAR images + GT masks  (label = 1)
#   • train/lookalike — look-alike SAR images + GT masks (label = 0)
#   • test/oil        — held-out oil images (inference only)
# NOTE: no_oil is NOT needed for Module 2 (bilge discriminator is oil vs lookalike).
# ─────────────────────────────────────────────────────────────────────────────
import glob, os, shutil
from pathlib import Path

INPUT_DIR = "/kaggle/input/datasets/rohithsheregar"
if not os.path.exists(INPUT_DIR):
    INPUT_DIR = "/kaggle/input/datasets" if os.path.exists("/kaggle/input/datasets") else "/kaggle/input"

print(f"📂 Input root: {INPUT_DIR}")
available_dirs = os.listdir(INPUT_DIR)
for d in sorted(available_dirs):
    print(f"   └─ {d}")

# ── Symlinks (oil + lookalike + test only) ────────────────────────────────────
WORKING_DATA = "/kaggle/working/data"
if os.path.exists(WORKING_DATA):
    shutil.rmtree(WORKING_DATA)

MAPPINGS = {
    "train/oil":       lambda n: "oil" in n and "spill" in n and "test" not in n,
    "train/lookalike": lambda n: "lookalike" in n and "test" not in n,
    "test/oil":        lambda n: "test" in n and "oil" in n,
}

total_tiffs = 0
print("\n🔗 Creating symlinks...")
for subpath, cond in MAPPINGS.items():
    matched = [d for d in available_dirs if cond(d.lower())]
    if not matched:
        # Fallback matching
        if "lookalike" in subpath:
            matched = [d for d in available_dirs if "lookalike" in d.lower()]
        elif "test" in subpath:
            matched = [d for d in available_dirs if "test" in d.lower()]
        else:
            matched = [d for d in available_dirs
                       if "oil" in d.lower() and "lookalike" not in d.lower()
                       and "no" not in d.lower() and "test" not in d.lower()]
    if not matched:
        print(f"   ⚠️  WARNING: no dataset matched for '{subpath}'")
        continue
    src = os.path.join(INPUT_DIR, matched[0])
    dst = os.path.join(WORKING_DATA, subpath)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    os.symlink(src, dst)
    n = len(glob.glob(os.path.join(src, "**", "*.tif*"), recursive=True))
    total_tiffs += n
    print(f"   ✅ {subpath} → {matched[0]}  ({n} TIFFs)")

print(f"\n📊 Total TIFFs: {total_tiffs}")

# ── Sanity check: discover scene pairs ───────────────────────────────────────
print("\n🔍 Dataset sanity check...")
from src.training.zenodo_sos_dataset import discover_sos_pairs

for cls in ["oil", "lookalike"]:
    cls_dir = Path(WORKING_DATA) / "train" / cls
    if cls_dir.exists():
        df = discover_sos_pairs(cls_dir, include_classes=[cls])
        n_with_mask = df["mask_path"].notna().sum() if "mask_path" in df.columns else "?"
        print(f"   ✅ {cls:10s}: {len(df)} scene pairs  ({n_with_mask} with GT masks)")
        if len(df) == 0:
            raise RuntimeError(f"❌ No {cls} pairs found! Check Kaggle dataset attachments.")
    else:
        raise RuntimeError(f"❌ {cls} directory missing: {cls_dir}")

print("\n✅ Cell 2 complete.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3 — DOWNLOAD MODULE 1 CHECKPOINT FROM HF HUB
# ─────────────────────────────────────────────────────────────────────────────
# Module 2 needs the Module 1 checkpoint to run on-the-fly SAR segmentation
# inference for any scene that doesn't have a ground-truth mask saved to disk.
#
# Priority 1: Download best_model.pt from Hugging Face Hub
# Priority 2: Check local Kaggle dataset input path
# Priority 3: No checkpoint — use GT masks only (skip scenes with missing masks)
# ─────────────────────────────────────────────────────────────────────────────
import os

PREFER_CHECKPOINT = "best_model.pt"   # or "last_model.pt"
DOWNLOAD_DIR      = "/kaggle/working/resume_ckpt"
os.makedirs(DOWNLOAD_DIR, exist_ok=True)

M1_CKPT = None

# ── Priority 1: Download from Hugging Face Hub ───────────────────────────────
if HF_REPO_ID and HF_TOKEN:
    print(f"🔍 Checking Hugging Face ({HF_REPO_ID}) for Module 1 checkpoints...")
    try:
        from huggingface_hub import HfFileSystem, hf_hub_download

        _fs    = HfFileSystem(token=HF_TOKEN)
        _files = _fs.ls(HF_REPO_ID, detail=False)

        # Look for .pt files at the repo root (not inside module2/ subfolder)
        _checkpoints = [f.split("/")[-1] for f in _files if f.endswith(".pt")]

        if _checkpoints:
            print(f"   Found {len(_checkpoints)} checkpoint(s) in HF Hub:")
            for _f in _checkpoints:
                print(f"     • {_f}")

            _target = PREFER_CHECKPOINT if PREFER_CHECKPOINT in _checkpoints else _checkpoints[-1]
            print(f"\n   ⬇️  Downloading '{_target}' from HF Hub...")
            _dest = hf_hub_download(
                repo_id   = HF_REPO_ID,
                filename  = _target,
                token     = HF_TOKEN,
                local_dir = DOWNLOAD_DIR,
            )
            print(f"   ✅ Downloaded to: {_dest}")
            M1_CKPT = _dest
        else:
            print("   ℹ️  No .pt files in HF Hub root — will use GT masks only.")
    except Exception as _e:
        print(f"   ⚠️  HF download failed: {_e}")

# ── Priority 2: Check Kaggle input datasets ──────────────────────────────────
if M1_CKPT is None:
    _kaggle_ckpt_paths = [
        "/kaggle/input/oil-spill-checkpoints/best_model.pt",
        "/kaggle/input/oil-spill-models/best_model.pt",
    ]
    for _p in _kaggle_ckpt_paths:
        if os.path.exists(_p):
            M1_CKPT = _p
            print(f"\n✅ Module 1 checkpoint found (Kaggle input): {M1_CKPT}")
            break

# ── Result ───────────────────────────────────────────────────────────────────
if M1_CKPT:
    import torch
    _ckpt  = torch.load(M1_CKPT, map_location="cpu", weights_only=False)
    _epoch = _ckpt.get("epoch", "?")
    _loss  = _ckpt.get("val_loss", "?")
    _miou  = _ckpt.get("val_miou", "?")
    print(f"\n▶  M1 Checkpoint  : {M1_CKPT}")
    print(f"   Saved epoch    : {_epoch}")
    print(f"   Best val_loss  : {_loss}")
    print(f"   Best mIoU      : {_miou}")
    print(f"\n🟢 M1 inference ACTIVE — scenes without GT masks will use M1 segmentation.")
else:
    print("\nℹ️  No Module 1 checkpoint found.")
    print("   Feature extraction will use GT masks only.")
    print("   Scenes with missing masks will be skipped.")

print("\n✅ Cell 3 complete.")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4 — MODULE 2 TRAINING
# ─────────────────────────────────────────────────────────────────────────────
# Runs the full Module 2 pipeline:
#   1. Feature extraction  (12 features per dark patch from oil + lookalike)
#   2. RandomForest training with GroupKFold CV (grouped by scene_id)
#   3. Bilge-dump filter (elongation > 3:1, area < 50 km², night boost +0.15)
#   4. Save model + metrics → upload to Hugging Face Hub
#
# ─── FEATURE CACHE ───────────────────────────────────────────────────────────
# If you're restarting after a timeout that completed feature extraction but
# not RF training, set USE_CACHED_FEATURES = True to skip re-extraction.
# The feature_summary.csv must exist from the prior run.
USE_CACHED_FEATURES = False    # ← set True to skip feature extraction on restart
# ─────────────────────────────────────────────────────────────────────────────
import os, time, torch

RESULTS_DIR   = "/kaggle/working/results/module2"
FEAT_CSV_PATH = f"{RESULTS_DIR}/metrics/feature_summary.csv"

# Auto-detect cache
if USE_CACHED_FEATURES and not os.path.exists(FEAT_CSV_PATH):
    print(f"⚠️  USE_CACHED_FEATURES=True but {FEAT_CSV_PATH} not found — will run fresh extraction.")
    USE_CACHED_FEATURES = False

# ── Build CLI flags ───────────────────────────────────────────────────────────
m1_flag     = f"--m1-checkpoint {M1_CKPT}"   if M1_CKPT         else ""
cache_flag  = "--cached-features"             if USE_CACHED_FEATURES else ""
hf_flags    = (
    f"--hf-repo-id {HF_REPO_ID} --hf-token {HF_TOKEN}"
    if HF_REPO_ID and HF_TOKEN else ""
)

# ── Session info ──────────────────────────────────────────────────────────────
print(f"⚡ GPU        : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"🧠 M1 ckpt   : {M1_CKPT or 'None (GT masks only)'}")
print(f"💾 Feat cache : {'✅ REUSING ' + FEAT_CSV_PATH if USE_CACHED_FEATURES else '🆕 Fresh extraction'}")
print(f"☁️  HF Hub    : {'ACTIVE → ' + HF_REPO_ID if HF_REPO_ID and HF_TOKEN else 'DISABLED'}")
print()

start = time.time()

!python -m src.training.train_module2 \
    --data-root /kaggle/working/data \
    --results-dir {RESULTS_DIR} \
    {m1_flag} \
    --gsd-m 10.0 \
    --n-folds 5 \
    --n-estimators 200 \
    --night-boost 0.15 \
    --min-elongation 3.0 \
    --max-area-km2 50.0 \
    {cache_flag} \
    {hf_flags}

elapsed = time.time() - start
print(f"\n🏁 Done in {elapsed/60:.1f} min")
print(f"📁 Local  : {RESULTS_DIR}/")
if HF_REPO_ID:
    print(f"☁️  HF Hub : https://huggingface.co/{HF_REPO_ID}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5 — UPLOAD TO HF HUB + SESSION SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
# This cell does TWO things:
#   1. Directly uploads all Module 2 output files to HuggingFace Hub
#      (this is the authoritative upload — train_module2.py's upload may have
#       been cut short by an OOM kernel restart, so this cell always re-uploads)
#   2. Prints a training summary from the saved metric files
# ═══════════════════════════════════════════════════════════════════════════════
import os, glob, shutil, json, time
from pathlib import Path
import pandas as pd

RESULTS_DIR = "/kaggle/working/results/module2"
OUT_DIR     = "/kaggle/working/session_output_m2"
os.makedirs(OUT_DIR, exist_ok=True)

# ── 5a. Collect files ─────────────────────────────────────────────────────────
print("Collecting Module 2 output files...")
COLLECT_PATTERNS = [
    f"{RESULTS_DIR}/checkpoints/*.joblib",
    f"{RESULTS_DIR}/metrics/*.csv",
    f"{RESULTS_DIR}/metrics/*.json",
    f"{RESULTS_DIR}/metrics/*.png",
]
collected = []
for pat in COLLECT_PATTERNS:
    for src in glob.glob(pat):
        dst     = shutil.copy(src, OUT_DIR)
        size_mb = os.path.getsize(dst) / (1024 ** 2)
        collected.append(src)
        print(f"   OK  {Path(src).name}  ({size_mb:.2f} MB)")

print(f"\nOutput directory: {OUT_DIR}")
print(f"Total files: {len(collected)}")

# ── 5b. Upload to HuggingFace Hub ─────────────────────────────────────────────
if not collected:
    print("\nNo output files found — did Cell 4 complete?")
elif HF_TOKEN and HF_REPO_ID:
    print(f"\nUploading {len(collected)} file(s) to HuggingFace Hub ({HF_REPO_ID})...")
    try:
        from huggingface_hub import HfApi
        _api = HfApi(token=HF_TOKEN)

        for src in collected:
            fname     = Path(src).name
            repo_path = f"module2/{fname}"
            _api.upload_file(
                path_or_fileobj = src,
                path_in_repo    = repo_path,
                repo_id         = HF_REPO_ID,
                commit_message  = f"Module2 auto-save: {fname}",
            )
            print(f"   HF uploaded: module2/{fname}")

        print(f"\nAll files at: https://huggingface.co/{HF_REPO_ID}")
        print("(Look in the module2/ folder of the repo)")
    except Exception as _hf_err:
        print(f"\nHF upload error: {_hf_err}")
        print("Files are still available in the Output tab.")
else:
    print("\nHF_TOKEN or HF_REPO_ID not set — files saved to Output tab only.")

# ── 5c. Training summary ──────────────────────────────────────────────────────
print(f"\n{'='*60}")
print("Module 2 Training Summary")
print(f"{'='*60}")

cv_path = f"{RESULTS_DIR}/metrics/cv_scores.json"
if os.path.exists(cv_path):
    with open(cv_path) as f:
        cv = json.load(f)
    summary = cv[-1]
    print(f"   {'CV balanced_accuracy':<28}: "
          f"{summary.get('mean_balanced_accuracy', float('nan')):.4f} "
          f"+/- {summary.get('std_balanced_accuracy', float('nan')):.4f}")
    print(f"   {'CV AUC':<28}: "
          f"{summary.get('mean_auc', float('nan')):.4f} "
          f"+/- {summary.get('std_auc', float('nan')):.4f}")
    print(f"   {'OOB score':<28}: {summary.get('oob_score', float('nan')):.4f}")

fi_path = f"{RESULTS_DIR}/metrics/feature_importance.csv"
if os.path.exists(fi_path):
    fi_df = pd.read_csv(fi_path)
    print(f"\n   Top-3 features by importance:")
    for _, r in fi_df.head(3).iterrows():
        print(f"     {r['feature']:<35}: {r['importance']:.4f} +/- {r['std']:.4f}")

tm_path = f"{RESULTS_DIR}/metrics/train_metrics.csv"
if os.path.exists(tm_path):
    tm = pd.read_csv(tm_path).iloc[0]
    print(f"\n   {'n_scenes':<28}: {tm.get('n_scenes', '?')}")
    print(f"   {'n_components':<28}: {tm.get('n_components', '?')}")
    print(f"   {'elapsed_s':<28}: {float(tm.get('elapsed_s', 0)):.0f}s")

print(f"\n{'='*60}")
print("\nModule 2 complete.")

---
## 📋 Module 2 Quick Reference

### What this notebook does
1. **12-feature extraction** per connected dark patch per scene:
   - *Polarimetric:* H, A (=0 dual-pol), α, VV/VH ratio
   - *Geometric:* area km², elongation, perimeter-area ratio, compactness
   - *Contextual:* wind speed, shipping-lane proximity, is_night
   - *Temporal:* morphology change km² (0.0 for single-pass)
2. **Random Forest (n=200, balanced_subsample)** — GroupKFold CV by scene_id
3. **Bilge-dump filter:** elongation > 3:1 AND area < 50 km² AND night-time boost +0.15

### Restart after timeout
If Cell 4 timed out after feature extraction but before RF training:
- Set `USE_CACHED_FEATURES = True` at the top of Cell 4
- Re-run Cell 4 — it skips the ~30 min extraction and goes straight to RF training

### Output files (all under `results/module2/`)
| File | Purpose |
|------|---------|
| `checkpoints/lookalike_rf.joblib` | Trained RF classifier |
| `metrics/cv_scores.json` | Per-fold balanced_accuracy + AUC + means |
| `metrics/feature_importance.png` | Bar chart of MDI importances |
| `metrics/feature_importance.csv` | Ranked feature importances |
| `metrics/feature_summary.csv` | Full extracted feature DataFrame (all scenes) |
| `metrics/detection_summary.csv` | Per-scene bilge candidate counts |
| `metrics/train_metrics.csv` | Module-level summary row |

### HF Hub structure
```
RohithSheregar/oil-spill-models/
├── best_model.pt          ← Module 1 segmentation model
├── last_model.pt          ← Module 1 last checkpoint
├── train_metrics.csv      ← Module 1 epoch metrics
└── module2/
    ├── lookalike_rf.joblib
    ├── cv_scores.json
    ├── feature_importance.png
    ├── feature_importance.csv
    ├── detection_summary.csv
    └── train_metrics.csv
```